# 1. Импорт библиотек и настройки

In [1]:
import pandas as pd

In [2]:
from pprint import pprint

In [3]:
pd.set_option('display.float_format', '{:.4f}'.format)

# 2. Загрузка констант и динамики

In [4]:
const = { 'Привлекаемые_средства': 380_000_000_000,
            'Ставка_купона_ОФЗ_ИН_л': 0.025,
            'Ставка_купона_ОФЗ_ПД': 0.1374,
            'Номинал_ОФЗ_ИН': 10_000,
            'Номинал_ОФЗ_ПД': 1000,
            'Количество_человек': 2_000_000,
            'НДФЛ': 0.13
        }
pprint(const)

{'Количество_человек': 2000000,
 'НДФЛ': 0.13,
 'Номинал_ОФЗ_ИН': 10000,
 'Номинал_ОФЗ_ПД': 1000,
 'Привлекаемые_средства': 380000000000,
 'Ставка_купона_ОФЗ_ИН_л': 0.025,
 'Ставка_купона_ОФЗ_ПД': 0.1374}


In [5]:
from cbr_inflation import get_inflation, get_latest_inflation, get_latest_target
from datetime import date
inf = get_inflation()
inf_d = inf.copy()
inf_d['Год'] = inf_d['date'].dt.year
inf_d.rename(columns={
    'inflation': 'Инфляция',
    'target': 'Цель по инфляции'},inplace=True)
inf_d = inf_d[['Год','Инфляция']]
inf_d = inf_d.tail(1)
current_year = date.today().year
forecast_years = [ current_year + 1, current_year + 2]
inf2 = pd.DataFrame({
    'Год': forecast_years,
    'Инфляция': inf['target'].iloc[-1]})
inf_res = pd.concat([inf_d,inf2],ignore_index=True)

In [6]:
import requests
import pandas as pd

BASE_URL = "http://www.cbr.ru/dataservice"

# Параметры для депозитов физических лиц
PUBLICATION_ID = 18      # В целом по РФ (депозиты)
DATASET_ID = 37
# Ставки по вкладам физических лиц

# Запрашиваем данные
params = {
    "publicationId": PUBLICATION_ID,
    "y1": 2020,
    "y2": 2026,
    "i_ids": [DATASET_ID],
    "m1_ids": [2],
    "m2_ids": [7]
}

response = requests.get(f"{BASE_URL}/dataEx", params=params)
data = response.json()
raw = data.get("RawData", [])

# Преобразуем в DataFrame
df = pd.DataFrame(raw)
# Переименуем колонки для удобства
df.rename(columns={
    'period': 'period_name',
    'date': 'date_str',
    'value': 'rate',
    'measure_1_id': 'currency_id',
    'measure_2_id': 'term_id',
    'period_id': 'period_id',
    'rowId': 'row_id'
}, inplace=True)
# Преобразуем дату в datetime
df['date'] = pd.to_datetime(df['date_str'], format='%d.%m.%Y')
df = df.sort_values('date').reset_index(drop=True)
# Добавим понятные названия для валют и сроков (можно будет подтянуть из /measures)
# Но для простоты оставим как есть

df = df[['date', 'rate', 'currency_id', 'term_id', 'period_name']]
sd = df.tail(1)
value = sd['rate'].iloc[0]

In [7]:
DEPOSIT_DECREMENT = 2.5
base = value
inf_res['Ставка депозита'] = base - (DEPOSIT_DECREMENT/100) * inf_res.index
inf_res[['Инфляция','Ставка депозита']] = inf_res[['Инфляция','Ставка депозита']]/100
inf_res

,Год,Инфляция,Ставка депозита
0,2026,0.0602,0.1284
1,2027,0.0400,0.1281
2,2028,0.0400,0.1279


# 3. ОФЗ ИН (л)

In [9]:
ofz =inf_res.copy()
ofz

,Год,Инфляция,Ставка депозита
0,2026,0.0602,0.1284
1,2027,0.0400,0.1281
2,2028,0.0400,0.1279


In [10]:
ofz['Привлекаемые средства'] = const['Привлекаемые_средства']
ofz['Количество человек'] = const['Количество_человек']
ofz['Ставка купона'] = const['Ставка_купона_ОФЗ_ИН_л']
ofz

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона
0,2026,0.0602,0.1284,380000000000,2000000,0.0250
1,2027,0.0400,0.1281,380000000000,2000000,0.0250
2,2028,0.0400,0.1279,380000000000,2000000,0.0250


In [11]:
ofz['На руках у человека, руб'] = ofz['Привлекаемые средства'] / ofz['Количество человек']
ofz

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб"
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000


In [12]:
ofz['Облигаций штук'] = ofz['На руках у человека, руб'] / const ['Номинал_ОФЗ_ИН']
ofz

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000


In [13]:
ofz['Инфляционный множитель']= (1 + inf_res['Инфляция']).cumprod()
ofz

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467


In [14]:
ofz['Номинал после индексации'] = const['Номинал_ОФЗ_ИН']*ofz['Инфляционный множитель']
ofz

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232


In [15]:
ofz['Номинал на начало'] = float(const['Номинал_ОФЗ_ИН'])
ofz.loc[ofz.index > 0, 'Номинал на начало'] = ofz['Номинал после индексации'].shift(1).fillna(const['Номинал_ОФЗ_ИН'])
ofz

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800


In [16]:
ofz['Индексация номинала'] = ofz['Номинал на начало'] * ofz['Инфляция']
ofz

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432


In [17]:
ofz['Купон, руб'] = ofz['Номинал после индексации'] * ofz['Ставка купона']
ofz

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб"
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000,265.0500
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800,275.6520
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432,286.6781


In [18]:
ofz['Доход без вычета'] = ofz['Купон, руб'] * ofz['Облигаций штук']
ofz

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000,265.0500,5035.9500
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800,275.6520,5237.3880
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432,286.6781,5446.8835


In [19]:
ofz ['Налоговый вычет, руб']= ofz['На руках у человека, руб'] * const['НДФЛ']
ofz

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета,"Налоговый вычет, руб"
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000,265.0500,5035.9500,24700.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800,275.6520,5237.3880,24700.0000
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432,286.6781,5446.8835,24700.0000


In [20]:
ofz['Доход с вычетом'] = ofz['Налоговый вычет, руб'] + ofz['Доход без вычета']
ofz

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета,"Налоговый вычет, руб",Доход с вычетом
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000,265.0500,5035.9500,24700.0000,29735.9500
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800,275.6520,5237.3880,24700.0000,29937.3880
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432,286.6781,5446.8835,24700.0000,30146.8835


In [21]:
cols = ofz.columns.tolist()
cols

['Год',
 'Инфляция',
 'Ставка депозита',
 'Привлекаемые средства',
 'Количество человек',
 'Ставка купона',
 'На руках у человека, руб',
 'Облигаций штук',
 'Инфляционный множитель',
 'Номинал после индексации',
 'Номинал на начало',
 'Индексация номинала',
 'Купон, руб',
 'Доход без вычета',
 'Налоговый вычет, руб',
 'Доход с вычетом']

In [22]:
df = ofz[['Год',
 'Привлекаемые средства',
 'Количество человек',
 'Инфляция',
 'Ставка купона',
 'На руках у человека, руб',
 'Облигаций штук',
 'Инфляционный множитель',
 'Номинал на начало',
 'Индексация номинала',
 'Номинал после индексации',
 'Купон, руб',
 'Доход без вычета',
 'Налоговый вычет, руб',
 'Доход с вычетом']]
df

,Год,Привлекаемые средства,Количество человек,Инфляция,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал на начало,Индексация номинала,Номинал после индексации,"Купон, руб",Доход без вычета,"Налоговый вычет, руб",Доход с вычетом
0,2026,380000000000,2000000,0.0602,0.0250,190000.0000,19.0000,1.0602,10000.0000,602.0000,10602.0000,265.0500,5035.9500,24700.0000,29735.9500
1,2027,380000000000,2000000,0.0400,0.0250,190000.0000,19.0000,1.1026,10602.0000,424.0800,11026.0800,275.6520,5237.3880,24700.0000,29937.3880
2,2028,380000000000,2000000,0.0400,0.0250,190000.0000,19.0000,1.1467,11026.0800,441.0432,11467.1232,286.6781,5446.8835,24700.0000,30146.8835


# 4. ОФЗ ПД

In [23]:
pprint (const)

{'Количество_человек': 2000000,
 'НДФЛ': 0.13,
 'Номинал_ОФЗ_ИН': 10000,
 'Номинал_ОФЗ_ПД': 1000,
 'Привлекаемые_средства': 380000000000,
 'Ставка_купона_ОФЗ_ИН_л': 0.025,
 'Ставка_купона_ОФЗ_ПД': 0.1374}


In [24]:
ofz_pd = ofz [['Год']].copy()
ofz_pd['Привлекаемые средства'] = const ["Привлекаемые_средства"]
ofz_pd ["Количество человек"] = const ["Количество_человек"]
ofz_pd ["Ставка купона"] = const ['Ставка_купона_ОФЗ_ПД']
ofz_pd ["На руках у человека"] = ofz [["На руках у человека, руб"]].copy()
ofz_pd ['Облигаций, штук'] = ofz_pd ['На руках у человека'] / const ['Номинал_ОФЗ_ПД']
ofz_pd ['Купон'] = const ['Номинал_ОФЗ_ПД'] * const ['Ставка_купона_ОФЗ_ПД']
ofz_pd ["Доход, руб"] = ofz_pd ["Купон"] * ofz_pd ['Облигаций, штук']
ofz_pd ['НДФЛ'] = ofz_pd ['Доход, руб'] * const ['НДФЛ']
ofz_pd ['Доход после вычета налога'] = ofz_pd['Доход, руб'] - ofz_pd ['НДФЛ']
ofz_pd

,Год,Привлекаемые средства,Количество человек,Ставка купона,На руках у человека,"Облигаций, штук",Купон,"Доход, руб",НДФЛ,Доход после вычета налога
0,2026,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
1,2027,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
2,2028,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200


# 5. Депозит

In [25]:
depozit = ofz[['Год']].copy()
depozit ['Привлекаемые средства'] = const ['Привлекаемые_средства']
depozit ['Количество человек'] = const ['Количество_человек']
depozit ['На руках у человека'] = ofz ['На руках у человека, руб']
depozit ['Ставка депозита'] = inf_res['Инфляция'] 


In [26]:
depozit ['Коэфициент'] = (1 + inf_res['Инфляция'] / 12) **12
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент
0,2026,380000000000,2000000,190000.0000,0.0602,1.0619
1,2027,380000000000,2000000,190000.0000,0.0400,1.0407
2,2028,380000000000,2000000,190000.0000,0.0400,1.0407


In [27]:
depozit ['Накопленный множитель'] = depozit ['Коэфициент'].cumprod()
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель
0,2026,380000000000,2000000,190000.0000,0.0602,1.0619,1.0619
1,2027,380000000000,2000000,190000.0000,0.0400,1.0407,1.1052
2,2028,380000000000,2000000,190000.0000,0.0400,1.0407,1.1502


In [28]:
 initial_amount = depozit['На руках у человека'].iloc[0]

In [29]:
depozit ['Сумма на конец года'] = initial_amount * depozit ['Накопленный множитель']
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года
0,2026,380000000000,2000000,190000.0000,0.0602,1.0619,1.0619,201758.9310
1,2027,380000000000,2000000,190000.0000,0.0400,1.0407,1.1052,209978.9011
2,2028,380000000000,2000000,190000.0000,0.0400,1.0407,1.1502,218533.7655


In [30]:
depozit ['Сумма на начало года'] = initial_amount
depozit.loc [depozit.index > 0, 'Сумма на начало года'] = depozit['Сумма на конец года']. shift (1)
# df_dep['Сумма на начало года '] = df_dep['Сумма на конец года'].shift(1).fillna(df_dep['На руках у человека, руб'].iloc[0])
# df_dep['Сумма начало'] = df_dep['Сумма конец'].shift(1).fillna(initial)
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года,Сумма на начало года
0,2026,380000000000,2000000,190000.0000,0.0602,1.0619,1.0619,201758.9310,190000.0000
1,2027,380000000000,2000000,190000.0000,0.0400,1.0407,1.1052,209978.9011,201758.9310
2,2028,380000000000,2000000,190000.0000,0.0400,1.0407,1.1502,218533.7655,209978.9011


In [31]:
depozit['Проценты']= depozit['Сумма на конец года'] - depozit['Сумма на начало года']
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года,Сумма на начало года,Проценты
0,2026,380000000000,2000000,190000.0000,0.0602,1.0619,1.0619,201758.9310,190000.0000,11758.9310
1,2027,380000000000,2000000,190000.0000,0.0400,1.0407,1.1052,209978.9011,201758.9310,8219.9701
2,2028,380000000000,2000000,190000.0000,0.0400,1.0407,1.1502,218533.7655,209978.9011,8554.8644


In [32]:
df_dep = depozit [["Год", "Привлекаемые средства", 
                   "Количество человек", 
                   "На руках у человека",
                   "Ставка депозита",
                   "Коэфициент",
                   "Накопленный множитель",
                   "Сумма на начало года",
                   "Сумма на конец года",
                    "Проценты"]]
df_dep          

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на начало года,Сумма на конец года,Проценты
0,2026,380000000000,2000000,190000.0000,0.0602,1.0619,1.0619,190000.0000,201758.9310,11758.9310
1,2027,380000000000,2000000,190000.0000,0.0400,1.0407,1.1052,201758.9310,209978.9011,8219.9701
2,2028,380000000000,2000000,190000.0000,0.0400,1.0407,1.1502,209978.9011,218533.7655,8554.8644


# 6. Доход за период (собрал без merge)

In [33]:
oin_summary = pd.DataFrame({
    'Инструмент': ['ОФЗ ИН (л)'],
    'Номинал': [ofz['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [ofz ['Индексация номинала'].sum()*ofz['Облигаций штук'].iloc[0] +  ofz['Доход без вычета'].sum()],
})
oin_summary ['Итоговая сумма'] = ofz['На руках у человека, руб'].iloc[0] + oin_summary['Доход, до налогов'] + ofz ['Налоговый вычет, руб'].iloc [0] 
oin_summary

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма
0,ОФЗ ИН (л),190000.0000,43595.5623,258295.5623


In [34]:
pd_summary = pd.DataFrame({
    'Инструмент': ['ОФЗ ПД'],
    'Номинал': [ofz ['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [ofz_pd['Доход, руб'].sum()],
    'Итоговая сумма': [ofz ['На руках у человека, руб'].iloc[0] + ofz_pd['Доход, руб'].sum()]})
pd_summary

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма
0,ОФЗ ПД,190000.0000,78318.0000,268318.0000


In [35]:
dep_summary = pd.DataFrame({
    'Инструмент': ['Депозит'],
    'Номинал': [ofz ['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [depozit['Проценты'].sum()],
    'Итоговая сумма': [ofz ['На руках у человека, руб'].iloc[0] + depozit['Проценты'].sum()]})
dep_summary

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма
0,Депозит,190000.0000,28533.7655,218533.7655


In [36]:
pohti_konec = pd.concat([oin_summary, pd_summary,dep_summary],ignore_index= True)
pohti_konec

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма
0,ОФЗ ИН (л),190000.0000,43595.5623,258295.5623
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000
2,Депозит,190000.0000,28533.7655,218533.7655


In [37]:
inf_factor = (1 + inf_res['Инфляция']).prod()

In [38]:
pohti_konec['Очистка инфляции'] = pohti_konec['Итоговая сумма']/inf_factor
pohti_konec['Реальный доход'] = pohti_konec['Итоговая сумма'] - pohti_konec['Номинал']
pohti_konec

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма,Очистка инфляции,Реальный доход
0,ОФЗ ИН (л),190000.0000,43595.5623,258295.5623,225248.7898,68295.5623
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000,233988.9398,78318.0000
2,Депозит,190000.0000,28533.7655,218533.7655,190574.1847,28533.7655


In [39]:
pohti_konec.loc[pohti_konec['Инструмент'] == 'ОФЗ ИН (л)', 'Очистка инфляции'] = None
# pohti_konec.loc[pohti_konec['Инструмент'] == 'ОФЗ ИН (л)', 'Реальный доход'] = None
pohti_konec

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма,Очистка инфляции,Реальный доход
0,ОФЗ ИН (л),190000.0000,43595.5623,258295.5623,NaN,68295.5623
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000,233988.9398,78318.0000
2,Депозит,190000.0000,28533.7655,218533.7655,190574.1847,28533.7655


# Делаем длинную таблицу (получилась широкая, но другая, будем думать дальше)

In [40]:
df_income = ofz[['Год', 'На руках у человека, руб']].copy()
df_income.rename(columns={'На руках у человека, руб': 'Вложения'}, inplace=True)
df_income

,Год,Вложения
0,2026,190000.0000
1,2027,190000.0000
2,2028,190000.0000


In [41]:
df_income['ОФЗ ИН доход'] = ofz['Индексация номинала']* ofz['Облигаций штук'] + ofz['Доход без вычета']
df_income

,Год,Вложения,ОФЗ ИН доход
0,2026,190000.0000,16473.9500
1,2027,190000.0000,13294.9080
2,2028,190000.0000,13826.7043


In [42]:
df_income = df_income.merge(ofz_pd[['Год', 'Доход, руб']], on='Год', how='left')
df_income.rename(columns={'Доход, руб': 'ОФЗ ПД доход'}, inplace=True)
df_income

,Год,Вложения,ОФЗ ИН доход,ОФЗ ПД доход
0,2026,190000.0000,16473.9500,26106.0000
1,2027,190000.0000,13294.9080,26106.0000
2,2028,190000.0000,13826.7043,26106.0000


In [43]:
df_income = df_income.merge(depozit[['Год', 'Проценты']], on='Год', how='left')
df_income.rename(columns={'Проценты': 'Депозит доход'}, inplace=True)
df_income

,Год,Вложения,ОФЗ ИН доход,ОФЗ ПД доход,Депозит доход
0,2026,190000.0000,16473.9500,26106.0000,11758.9310
1,2027,190000.0000,13294.9080,26106.0000,8219.9701
2,2028,190000.0000,13826.7043,26106.0000,8554.8644


In [44]:
# строка итого
total_row = df_income[['ОФЗ ИН доход', 'ОФЗ ПД доход', 'Депозит доход']].sum()
total_row['Год'] = 'Итого'
total_row['Вложения'] = df_income['Вложения'].iloc[0]  # начальные вложения (одинаковы)
df_income = pd.concat([df_income, pd.DataFrame([total_row])], ignore_index=True)
df_income

,Год,Вложения,ОФЗ ИН доход,ОФЗ ПД доход,Депозит доход
0,2026,190000.0000,16473.9500,26106.0000,11758.9310
1,2027,190000.0000,13294.9080,26106.0000,8219.9701
2,2028,190000.0000,13826.7043,26106.0000,8554.8644
3,Итого,190000.0000,43595.5623,78318.0000,28533.7655


In [45]:
inf_factor_odin = (1 + inf_res['Инфляция'])


# Делаем длинную 


In [46]:
# 1. Убираем строку 'Итого' из df_income
df_income_without_total = df_income[df_income['Год'] != 'Итого']

# 2. Теперь применяем melt к данным без итогов
df_long = df_income_without_total.melt(
    id_vars=['Год', 'Вложения'],
    value_vars=['ОФЗ ИН доход', 'ОФЗ ПД доход', 'Депозит доход'],
    var_name='Инструмент',
    value_name='Доход'
)

# 3. Сортируем по году и инструменту
df_long = df_long.sort_values(['Год', 'Инструмент']).reset_index(drop=True)

df_long

,Год,Вложения,Инструмент,Доход
0,2026,190000.0000,Депозит доход,11758.9310
1,2026,190000.0000,ОФЗ ИН доход,16473.9500
2,2026,190000.0000,ОФЗ ПД доход,26106.0000
3,2027,190000.0000,Депозит доход,8219.9701
4,2027,190000.0000,ОФЗ ИН доход,13294.9080
5,2027,190000.0000,ОФЗ ПД доход,26106.0000
6,2028,190000.0000,Депозит доход,8554.8644
7,2028,190000.0000,ОФЗ ИН доход,13826.7043
8,2028,190000.0000,ОФЗ ПД доход,26106.0000


In [47]:
ofz_pd

,Год,Привлекаемые средства,Количество человек,Ставка купона,На руках у человека,"Облигаций, штук",Купон,"Доход, руб",НДФЛ,Доход после вычета налога
0,2026,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
1,2027,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
2,2028,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
